# InduSense — Sprint 1 : audit des données (journal de bord)

**Compétences** : C1 (jeu de données pertinent et cohérent), C2 (risques éthiques et réglementaires),
C3 (préparation et intégrité des données).

Ce notebook ne contient **pas** la logique de traitement : elle vit dans le package `src/indusense`
(testé, réutilisable, piloté par `configs/`). Ici, on exécute, on observe, on **interprète** et on **décide**.
Chaque décision est reportée dans le contrat de données correspondant (`configs/contracts/`).

| Étape | Commande équivalente | Artefacts |
|---|---|---|
| 1. Cartes d'identité | `uv run indusense identity` | `artifacts/ingestion/identities/` |
| 2. Audit + cartographie | `uv run indusense audit` | `artifacts/ingestion/audits/` |
| 3. Brouillons de contrats | `uv run indusense contract draft` | `artifacts/ingestion/contracts/drafts/` |
| 4. Contrôle des contrats | `uv run indusense validate` | `artifacts/ingestion/validations/` |

## 0. Mise en place

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from indusense.config import all_sources, load_settings
from indusense.pipeline import run_pipeline


def project_root(start: Path = Path.cwd()) -> Path:
    """Remonte l'arborescence jusqu'au dossier qui contient configs/pipeline.yaml."""
    for p in [start, *start.parents]:
        if (p / "configs" / "pipeline.yaml").exists():
            return p
    raise FileNotFoundError("configs/pipeline.yaml introuvable")


ROOT = project_root()
settings = load_settings(ROOT / "configs" / "pipeline.yaml")
pd.DataFrame([{"id": s.id, "fichier": s.path.name, "format": s.resolved_format,
               "existe": s.path.exists()} for s in all_sources(settings)])

## 1. Cartes d'identité (TP1)

Un fichier par source dans `artifacts/ingestion/identities/AAAAMMJJ/<fichier>_HHMMSS.md`.

In [ ]:
run = run_pipeline(settings, ["identity", "audit"])
for out in run.outputs:
    if "/identities/" in out:
        display(Markdown((ROOT / out).read_text(encoding="utf-8")))

## 2. Audit qualité

Les constats sont classés par **dimension de qualité** (guide `docs/methodo/data_audit.md`).
🔴 bloquant · 🟠 à traiter · 🔵 information.

In [ ]:
from indusense.io.readers import load_source
from indusense.profiling.profile import profile_dataset

profiles = {}
for spec in all_sources(settings):
    for ds in load_source(spec, settings).datasets:
        profiles[ds.id] = profile_dataset(ds, settings)

synthese = pd.DataFrame([{
    "jeu": p.dataset_id, "lignes": p.n_rows, "colonnes": p.n_cols,
    "doublons": p.n_duplicate_rows,
    "🔴": sum(f.severity == "error" for f in p.findings),
    "🟠": sum(f.severity == "warning" for f in p.findings),
    "🔵": sum(f.severity == "info" for f in p.findings)} for p in profiles.values()])
synthese

In [ ]:
constats = pd.DataFrame([{"jeu": p.dataset_id, **vars(f)}
                         for p in profiles.values() for f in p.findings])
constats.sort_values(["severity", "jeu", "dimension"]) if not constats.empty else "Aucun constat"

### Interprétation des constats

Pour chaque constat 🔴/🟠, une cellule sur ce modèle. **C'est la partie évaluée** : le raisonnement,
pas le code. Exemple à adapter à tes résultats réels :

> **Constat** : `telemetry.temperature_c` — min 32.00 et max 80.00, valeurs rondes répétées.
> **Observation** : N mesures exactement à 32.00 et M à 80.00 sur 135 626 lignes.
> **Hypothèse** : plage de mesure du capteur écrêtée (valeurs réelles au-delà non enregistrées).
> **Vérification** : fiche technique du capteur / responsable OT.
> **Décision** : Bronze : conserver tel quel. Silver : ajouter un indicateur `temperature_saturee`.
> Contrat : bornes physiques `min: 0`, `max: 120` (`# TODO valider`).
> **Justification** : une saturation peut précéder une panne (signal utile) ; supprimer ces lignes
> biaiserait le modèle vers les régimes normaux.

> **Constat** :
> **Observation** :
> **Hypothèse** :
> **Vérification** :
> **Décision** :
> **Justification** :

## 3. Cartographie

Relations déclarées dans les contrats (`references`) et détectées automatiquement (colonnes de même nom).
Le diagramme Mermaid s'affiche dans GitHub et dans l'aperçu Markdown de VS Code
(extension *Markdown Preview Mermaid Support*).

In [ ]:
carto = next(o for o in run.outputs if "cartographie" in o and o.endswith(".md"))
display(Markdown((ROOT / carto).read_text(encoding="utf-8")))

## 4. Contrats de données

Le brouillon décrit l'**observé** ; le contrat de `configs/contracts/` décrit l'**attendu**
(bornes métier, valeurs autorisées, décisions RGPD). Skill de référence : `data-contract`.

In [ ]:
run_validation = run_pipeline(settings, ["draft", "validate"])
rapport = next(o for o in run_validation.outputs if "/validations/" in o and o.endswith(".md"))
display(Markdown((ROOT / rapport).read_text(encoding="utf-8")))

### Décisions sur les violations

| Violation | Qualification (donnée fautive / contrat faux / tolérable) | Action | Couche |
|---|---|---|---|
| | | | |

## 5. Synthèse pour la datasheet

- **Pertinence pour le cas d'usage (C1)** :
- **Risques et conformité (C2)** : données personnelles, biais de représentativité :
- **Préparation à prévoir (C3)** : règles Silver issues des décisions ci-dessus :
- **Limites de l'audit** :

Compléter `docs/datasheets/<jeu>.md` à partir de `docs/datasheets/TEMPLATE.md`.